# Chặng 6 - Thực nghiệm: Hệ thống Phân loại Cảm xúc tiếng Việt (Sentiment Analysis)

**Trường Đại học Nam Cần Thơ - Môn: Máy học nâng cao (25MIT-1A)**
**Giảng viên:** TS. Huỳnh Văn Huy
**Nhóm:** Võ Khương Duy (2513464) - Nguyễn Thị Mai Hân (2513562) - Nguyễn Minh Nhựt (2513525)

Notebook này **chỉ clone mã nguồn từ GitHub và gọi lệnh python** - toàn bộ logic (tiền xử lý, baseline, fine-tune, đánh giá, web demo) nằm trong repo `code_giua_ki_train_AI`.

```text
Clone repo -> python scripts/run_pipeline.py   (toàn bộ thực nghiệm)
          -> python scripts/demo_inference.py  (demo "Sản phẩm rất tệ!")
          -> python webapp/run_web.py          (web demo Flask + link public)
```

## Quy trình tổng quát (9 bước)

```text
UIT-VSFC (3 lớp) -> Tiền xử lý -> Tokenizer PhoBERT -> Train/Valid/Test
   -> (1) TF-IDF + Logistic Regression (baseline)
   -> (2) PhoBERT fine-tuned sẵn (đối chứng, không huấn luyện)
   -> (3) Fine-tuning PhoBERT-base-v2 (GPU Colab T4)
   -> Đánh giá (Accuracy/P/R/F1/CM/PR curve) -> Lưu model local
   -> Flask web (Tailwind) + Cloudflared tunnel -> demo public
```

## 1. Thiết lập môi trường

In [ ]:
# Kiểm tra GPU (Colab free: Tesla T4)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cài đặt thư viện
!pip install -q transformers scikit-learn matplotlib seaborn pandas numpy huggingface_hub flask requests cloudflared


## 2. Clone mã nguồn mới nhất từ GitHub

Cell dưới sẽ **xóa clone cũ (nếu có) rồi clone bản mới** về `code_giua_ki_train_AI/`.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/duyvo26/code_giua_ki_train_AI"
PROJECT_DIR = "code_giua_ki_train_AI"

# Xoá clone cũ để luôn dùng bản code mới nhất từ GitHub
if os.path.isdir(PROJECT_DIR):
    print("-> Xoa clone cu...")
    !rm -rf {PROJECT_DIR}

print("-> Clone code moi tu GitHub...")
!git clone --depth 1 {REPO_URL}
print("-> Clone xong")

PROJECT_DIR = os.path.abspath(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print("-> Thu muc lam viec:", PROJECT_DIR)


## 3. Chạy pipeline thực nghiệm

Lệnh `python scripts/run_pipeline.py` thực hiện toàn bộ:

```text
Bước 1-3: Tải UIT-VSFC + tiền xử lý + gán nhãn (0=Neg, 1=Neu, 2=Pos)
Thực nghiệm 1: TF-IDF + Logistic Regression (baseline)
Thực nghiệm 2: PhoBERT fine-tuned sẵn (wonrax - đối chứng, không train)
Thực nghiệm 3: Fine-tuning PhoBERT-base-v2 (~15-20 phút trên T4)
Bước 7-8: Đánh giá (Acc/P/R/F1/CM/PR curve) + bảng so sánh -> results/
```

> Chạy lần đầu đầy đủ. Nếu session Colab bị mất mà model đã có sẵn, chạy lại với `--skip-finetune` để không retrain.

In [ ]:
# Chạy toàn bộ pipeline (fine-tune ~15-20 phút)
!python scripts/run_pipeline.py


## 4. Demo inference (Bước 9)

Dự báo cảm xúc cho `"Sản phẩm rất tệ!"` và các câu mẫu bằng model đã fine-tune.

In [ ]:
# Demo: in nhãn + xác suất % 3 lớp cho 5 câu mẫu
!python scripts/demo_inference.py


## 5. Web demo - Flask + Cloudflared tunnel

`python webapp/run_web.py` khởi động Flask web (giao diện Tailwind) và mở **Cloudflared tunnel** - in ra link public `https://xxx.trycloudflare.com` mở trên trình duyệt bất kỳ.

Giao diện gồm 3 chức năng:
- **Thông tin model**: Accuracy, F1-macro, Recall lớp Negative, thời gian train.
- **Train lại**: fine-tune chạy thread nền, hiển thị tiến trình epoch, không treo web.
- **Dự đoán cảm xúc**: nhập bình luận -> nhãn + xác suất % 3 lớp.

> Cell này chạy mãi (giữ tunnel sống). Dừng web khi hết demo: nhấn **dừng cell** (biểu tượng stop).

In [ ]:
# Chạy web demo + tunnel (giữ cell chạy để link còn hoạt động)
!python webapp/run_web.py


## 6. (Tuỳ chọn) Lưu kết quả lên Google Drive

Sao lưu `results/` và `models/` về Drive để phục vụ báo cáo và triển khai nội bộ.

In [ ]:
# Chỉ chạy nếu muốn lưu sang Drive
from google.colab import drive
import shutil

try:
    drive.mount("/content/drive")
    out_dir = "/content/drive/MyDrive/code_giua_ki_train_AI"
    shutil.copytree("results", f"{out_dir}/results", dirs_exist_ok=True)
    shutil.copytree("models", f"{out_dir}/models", dirs_exist_ok=True)
    print("Da luu ket qua + mo hinh vao Drive:", out_dir)
except Exception as exc:
    print("Bo qua (khong mount Drive):", exc)


## Kết luận & việc cần làm tiếp theo

1. Điền kết quả từ `results/compare_table.md` vào bảng so sánh trong báo cáo.
2. Phân tích confusion matrix + PR curve, đặc biệt **Recall lớp Negative** (bao nhiêu phàn nàn bị bỏ sót).
3. Chụp ảnh web demo (cell 5): thông tin model + dự đoán `"Sản phẩm rất tệ!"`.
4. Triển khai nội bộ (không cần tunnel): `python webapp/run_web.py --no-tunnel` rồi mở `http://localhost:8080`.
5. Bảo mật: model + inference chạy 100% cục bộ - dữ liệu bình luận không gửi ra ngoài (Cloudflared chỉ là đường ống web).

> **Lưu ý khoa học:** không cam kết độ chính xác >95% như đề bài gợi ý. Báo cáo số liệu THỰC tế đo được trên tập test và diễn giải (Accuracy, F1, Recall lớp Negative).